# Zero-Shot Visual Agent
BLIP (VQA) · OWL-ViT (open-vocabulary detection) · CLIP (zero-shot classification) · ResNet-50 (CNN baseline), orchestrated by a **LangGraph** agent with a verification loop.

The logic lives in the `agent/` package so the notebook, the Streamlit app and the evaluation scripts all share the same code. Run this notebook from the project folder.

In [ ]:
# Setup (run once; skip on a machine where requirements.txt is already installed)
# %pip install -q -r requirements.txt
import sys, random, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from PIL import Image

PROJECT_DIR = Path.cwd()
assert (PROJECT_DIR / "agent").exists(), "Open the notebook from the project folder (the one containing agent/)."
sys.path.insert(0, str(PROJECT_DIR))
random.seed(42); np.random.seed(42)

# Point this at any folder of images (Kaggle example shown). Falls back to a manual path.
DEMO_IMAGE_DIR = Path("/kaggle/input/datasets/bhuvan181/sampleeee")
MANUAL_IMAGE = None   # e.g. Path("my_photo.jpg")

In [ ]:
from agent import VisualAgent, LANGGRAPH_AVAILABLE
from agent.models import ModelHub
from agent.utils import prepare_image, annotate, save_run

hub = ModelHub()          # models are loaded lazily, only when a tool needs them
agent = VisualAgent(hub)
print("Device:", hub.device, "| orchestration:", agent.backend)

In [ ]:
exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
files = sorted(p for p in DEMO_IMAGE_DIR.rglob("*") if p.suffix.lower() in exts) if DEMO_IMAGE_DIR.exists() else []
IMAGE_PATH = MANUAL_IMAGE or (files[0] if files else None)
assert IMAGE_PATH, "Set MANUAL_IMAGE or DEMO_IMAGE_DIR to an image."
image = prepare_image(IMAGE_PATH)
print(IMAGE_PATH, image.size)
display(image)

## 1. General question
The agent plans which tools to run, cross-checks the VQA answer with the detector, then composes an answer.

In [ ]:
result = agent.run(image, "What objects are visible in this image?")
display(Markdown(result["summary"]))
print("Agreement:", result["agreement"], "| plan:", result["plan"])
display(pd.DataFrame(result["trace"]))
print(result["answer"])

In [ ]:
dets = (result["tool_results"].get("detection") or {}).get("detections", [])
display(annotate(image, dets))

## 2. Counting (answered with the detector, not BLIP)

In [ ]:
r = agent.run(image, "How many people are there?")
display(Markdown(r["summary"])); print(r["uncertainty"])

## 3. Safety screening
Paired hazard/safe CLIP prompts, per-person PPE crops (only applied on work-site scenes) and box-geometry proximity.

In [ ]:
r = agent.run(image, "Are there any potential safety hazards?", {"mode": "safety"})
s = r["tool_results"]["safety"]
print("Overall:", s["overall"], "| work-site probability:", s["worksite_probability"])
display(pd.DataFrame(s["findings"])); display(pd.DataFrame(s["scene_checks"]))
print(s["warning"])

## 4. Zero-shot classification with custom labels

In [ ]:
r = agent.run(image, "What kind of scene is this?", {"custom_labels": ["city street", "forest", "kitchen", "beach"]})
display(pd.DataFrame(r["tool_results"]["clip"]["rows"])); display(pd.DataFrame(r["tool_results"]["cnn"]["rows"]))

In [ ]:
# Log the run (same JSONL file the Streamlit History tab reads)
print("Saved to", save_run(IMAGE_PATH, "What objects are visible in this image?", result))

## 5. Evaluation
**CIFAR-10:** CLIP zero-shot vs. ResNet-50 (ImageNet classes mapped to CIFAR). Pure zero-shot: CLIP never sees CIFAR labels.

**Your own images:** copy `evaluation/evaluation_template.csv`, add human-labelled rows, then run `evaluation/evaluate_agent.py` (see README). Do not report hazard accuracy until you have human-reviewed labels.

In [ ]:
# Downloads CIFAR-10 via torchvision. On Kaggle add:  --kaggle-dir /kaggle/input/competitions/cifar-10
!python evaluation/evaluate_cifar10.py --n 1000

## 6. Frontend
Locally run `streamlit run app.py` (or double-click `run.bat` / `run.sh`).